# q-0001 — DAG assessment: the August 2026 PWA conversion-rate flip

**Question, verbatim:**

> in relation to the change in price configuration for pwa in august 2026, no random
> experiments were conducted, how can i estimate the impact of the change in terms of
> booking volume?

**`data_mode: simulated`. NO NUMBER IN THIS NOTEBOOK SAYS ANYTHING ABOUT IAGL.**
Every figure below is about a synthetic world built from the graph in `wiki/concepts/`.
The notebook exists to answer three things: does the graph as drawn even generate
coherent data, does the proposed estimator recover an effect we planted, and how large
would an effect have to be before this design could see it. `observed:` flags on the
nodes are design choices for this simulation, not verified facts about the warehouse.

**Reviewer verdict, quoted (`wiki/questions/q-0001/review.md`):**

> **IDENTIFIED UNDER STATED ASSUMPTIONS — and the assumption doing all the work is not
> one this graph can support.** Adjustment cannot identify this effect: the only backdoor
> path runs through `departure_year_within_2026`, which *is* the assignment rule, so
> conditioning on it destroys all treatment variation — positivity fails exactly where the
> confounding lives. `booking_departure_lead_time` must be excluded from every adjustment
> set for the same reason. DiD is therefore the right family, but parallel trends is a
> restriction on trends, not on edges, so this DAG can neither bless nor refuse it; it must
> be earned empirically, and one month of pre-period does not earn it.

**Adjustment set: empty, deliberately.** The graph says so, and this is the single most
important line in the notebook. The only backdoor into the treatment runs through
`departure_year_within_2026`, which *is* the assignment rule; conditioning on it removes
all treatment variation. `booking_departure_lead_time` is a deterministic descendant of
the same boundary and is excluded for the same reason (reviewer requirement 7).
Identification comes from the *design* — differencing across the departure-year boundary
over time — not from covariate adjustment.

**Design being tested:** difference-in-differences across the departure-date boundary.
Treated = bookings departing within 2026; control = bookings departing 2027+. Time
dimension is booking date. Treatment window 1–14 Aug 2026. On a **log/Poisson scale**
(reviewer requirement 3), with an **18-month pre-period** (requirement 2), an event study
for anticipation, a **stated MDE before the headline** (requirement 1) and a **placebo
battery reported before the estimate** (requirement 6).

**Events overlapping the window and how each is handled**

| event | handling |
|---|---|
| The flip itself, 1–14 Aug 2026, departures within 2026 only | the treatment |
| Unnamed 2025 change contaminating Aug 2025 | the year-on-year DiD is **not used**; a same-period-last-year comparison is ruled out up front as already-known-biased {source: raw/pwa_exp} |
| Announcement date unknown — members may have anticipated | tested, not assumed: event-study leads before 1 Aug and lags after 14 Aug |
| Nothing else, per the analyst | a stated belief from one person, not an audit — the placebo-window battery tests for unexplained level shifts anyway |

**Assumed and not checkable from data**

1. **Parallel trends** between 2026-departure and 2027+-departure bookings. Untestable in
   the post-period by construction. Pre-period fit is evidence, never proof.
2. **No pull-forward** across the boundary — the analyst ruled out members moving departure
   dates to chase the better rate {by:analyst on:2026-09-07}. If wrong, the estimate is
   inflated from both sides at once.
3. **No spillover** from treated to control bookings.
4. `seasonal_holiday_demand` is unobserved. DiD absorbs it **only if it moves both
   departure-year groups together**. If it moves them differently in that fortnight, the
   design is broken and nothing here detects it.

**Prior used:** none. `wiki/experiments/2026-08-conversion-rate-flip.md` deliberately
records no readout, so there is no prior in the building to anchor on.

**Skills used:** none — no access or method skill exists yet.

Dependencies: `numpy`, `pandas`, `statsmodels`, `matplotlib`, `networkx`. Nothing exotic,
no network calls, no imports from the cb repository.

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

SEED = 20260907
rng = np.random.default_rng(SEED)

# The effect we PLANT. The whole point of a simulated notebook is to check the estimator
# gets this back. 0.08 = an 8% lift in member booking volume for treated bookings during
# the flip window. It is a made-up number and is not a belief about IAGL.
TRUE_LIFT = 0.08

# How much of the observed move is pure substitution (members switching INTO redeeming on
# bookings they would have made anyway). This is the failure mode the outcome choice
# exists to catch, so we plant a large one.
TRUE_SUBSTITUTION = 0.35

print(f"seed = {SEED}")
print(f"PLANTED true lift on member_booking_volume = {TRUE_LIFT:+.1%}")
print(f"PLANTED substitution into redeeming        = {TRUE_SUBSTITUTION:+.1%}")
print("\nSIMULATED DATA. Nothing below is a statement about IAGL.")

## 1. The graph, exactly as `wiki/concepts/` draws it

Every edge below is a line in a file. If the generator in the next cell cannot be written
from this graph, the graph is incoherent and that is a finding in itself.

In [ ]:
# Causal edges only. member_booking_volume = redeeming + non-redeeming is ARITHMETIC
# (## Computed from) and is deliberately NOT an edge here.
EDGES = [
    ("departure_year_within_2026", "pwa_conversion_rate_flip"),
    ("booking_in_flip_window",     "pwa_conversion_rate_flip"),
    ("departure_year_within_2026", "booking_departure_lead_time"),
    ("booking_departure_lead_time", "member_booking_volume"),
    ("booking_departure_lead_time", "avios_balance"),
    ("avios_balance",              "avios_redeeming_bookings"),
    ("seasonal_holiday_demand",    "member_booking_volume"),
    ("seasonal_holiday_demand",    "avios_redeeming_bookings"),
    ("pwa_conversion_rate_flip",   "avios_redeeming_bookings"),   # mechanism, confirmed
    ("pwa_conversion_rate_flip",   "member_booking_volume"),      # THE HYPOTHESIS
]

OBSERVED = {
    "departure_year_within_2026": True,
    "booking_in_flip_window": True,
    "pwa_conversion_rate_flip": True,
    "booking_departure_lead_time": True,
    "avios_balance": True,     # UNVERIFIED against live systems — reviewer requirement 9
    "seasonal_holiday_demand": False,
    "avios_redeeming_bookings": True,
    "member_booking_volume": True,
}

G = nx.DiGraph(EDGES)
assert nx.is_directed_acyclic_graph(G), "the graph as drawn contains a cycle"
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges, acyclic: OK")
print("unobserved:", [n for n, o in OBSERVED.items() if not o])

pos = nx.spring_layout(G, seed=3)
colours = ["#dddddd" if OBSERVED[n] else "#ff9999" for n in G.nodes]
plt.figure(figsize=(11, 7))
nx.draw(G, pos, with_labels=True, node_color=colours, node_size=2600,
        font_size=8, arrowsize=16, edgecolors="black")
plt.title("q-0001 pwa-pricing graph (red = unobserved)")
plt.show()

## 2. Why the adjustment set is empty — shown, not asserted

Conditioning on the assignment rule kills the variation. This cell demonstrates that
rather than taking the reviewer's word for it.

In [ ]:
T, Y = "pwa_conversion_rate_flip", "member_booking_volume"

print("parents of the treatment:", sorted(G.predecessors(T)))
print("parents of the outcome:  ", sorted(G.predecessors(Y)))
print()
print("Backdoor paths from treatment to outcome, ignoring edge direction:")
U = G.to_undirected()
for path in nx.all_simple_paths(U, T, Y):
    if len(path) > 2 and not G.has_edge(path[0], path[1]):   # leaves T against the arrow
        print("   ", " - ".join(path))
print()
print("Every one of them starts with departure_year_within_2026, which IS the assignment")
print("rule. Conditioning on it leaves no treated/untreated contrast within a stratum:")
print("positivity fails exactly where the confounding lives. booking_departure_lead_time")
print("is a deterministic descendant of the same boundary and is excluded too.")
print()
print("=> ADJUSTMENT SET = {} (empty). Identification is from the DESIGN, not covariates.")

## 3. The generator — one line per edge

Panel at **booking-date × departure-year-group**, daily, from Feb 2025 to Sep 2026: an
18-month pre-period as the reviewer required, so parallel trends can actually be tested
rather than asserted.

The simulation deliberately makes life hard: the two groups have **different levels,
different seasonal amplitudes and different trends**, because the analyst said 2027
departures are "a different kind of trip". A generator with no confounding would prove
the estimator works in a world we already know we are not in.

In [ ]:
START, END = pd.Timestamp("2025-02-01"), pd.Timestamp("2026-09-30")
FLIP_START, FLIP_END = pd.Timestamp("2026-08-01"), pd.Timestamp("2026-08-14")
dates = pd.date_range(START, END, freq="D")

# Set to True to break parallel trends on purpose and see whether the checks below catch
# it. Leave False for the main run.
BREAK_PARALLEL_TRENDS = False

def simulate(true_lift=TRUE_LIFT, break_pt=BREAK_PARALLEL_TRENDS, rng=rng):
    rows = []
    doy = dates.dayofyear.values
    t = (dates - START).days.values / 365.0

    # seasonal_holiday_demand: UNOBSERVED common driver. Shared component + a group-specific
    # amplitude. DiD survives this as long as the shared part moves both groups together.
    shared_season = 0.22 * np.sin(2 * np.pi * (doy - 20) / 365.0)
    shock = rng.normal(0, 0.05, len(dates))            # common demand shocks

    for group, is_2026 in [("depart_2026", 1), ("depart_2027plus", 0)]:
        # departure_year_within_2026 -> booking_departure_lead_time (deterministic-ish)
        lead_days = 150 if is_2026 else 480
        # booking_departure_lead_time -> member_booking_volume: different level, different
        # seasonal amplitude, different trend. This is the threat, planted.
        base = np.log(1800) if is_2026 else np.log(260)
        amp = 1.0 if is_2026 else 0.55
        trend = 0.06 if is_2026 else 0.02
        if break_pt:
            trend += 0.25 * is_2026        # a divergence DiD cannot absorb

        # pwa_conversion_rate_flip: booking_in_flip_window AND departure_year_within_2026
        treated = ((dates >= FLIP_START) & (dates <= FLIP_END)).astype(int) * is_2026

        log_mu = base + amp * shared_season + trend * t + shock + np.log1p(treated * true_lift)
        bookings = rng.poisson(np.exp(log_mu))

        # avios_balance -> avios_redeeming_bookings, plus the substitution the flip causes.
        # This is a SPLIT of bookings, not extra bookings: arithmetic, not cause.
        base_share = 0.30 if is_2026 else 0.26
        share = np.clip(base_share + treated * TRUE_SUBSTITUTION, 0, 1)
        redeeming = rng.binomial(bookings, share)

        rows.append(pd.DataFrame({
            "booking_date": dates,
            "departure_group": group,
            "treated_group": is_2026,
            "in_window": ((dates >= FLIP_START) & (dates <= FLIP_END)).astype(int),
            "treated": treated,
            "booking_departure_lead_time": lead_days,
            "member_booking_volume": bookings,
            "avios_redeeming_bookings": redeeming,
        }))
    df = pd.concat(rows, ignore_index=True)
    df["non_redeeming_bookings"] = df.member_booking_volume - df.avios_redeeming_bookings
    return df

df = simulate()
print(df.groupby("departure_group").member_booking_volume.agg(["count", "mean", "std"]))
df.head()

## 4. Assertions — before anything is estimated

These catch the errors that complete silently and return plausible numbers.

In [ ]:
checks = {}

def check(name, cond, msg):
    checks[name] = bool(cond)
    assert cond, msg

check("grain_unique",
      not df.duplicated(["booking_date", "departure_group"]).any(),
      "grain broken — more than one row per booking_date x departure_group (join fanned out)")
check("balanced_panel",
      df.groupby("departure_group").size().nunique() == 1,
      "unbalanced panel — the two groups do not cover the same dates")
check("no_nulls",
      df.isna().sum().sum() == 0,
      "unexpected nulls in the panel")
check("arithmetic_holds",
      (df.avios_redeeming_bookings + df.non_redeeming_bookings == df.member_booking_volume).all(),
      "member_booking_volume != redeeming + non-redeeming — the Computed-from identity is broken")
check("redeeming_is_subset",
      (df.avios_redeeming_bookings <= df.member_booking_volume).all(),
      "redeeming bookings exceed total member bookings — subset relation violated")
check("control_never_treated",
      df.loc[df.departure_group == "depart_2027plus", "treated"].sum() == 0,
      "control group shows treatment — the assignment rule is wrong in the generator")
check("treatment_window_is_14_days",
      df.loc[df.treated == 1, "booking_date"].nunique() == 14,
      "treated period is not 14 days")
check("preperiod_at_least_18_months",
      (FLIP_START - df.booking_date.min()).days >= 540,
      "pre-period shorter than 18 months — reviewer requirement 2 not met")

for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")

## 5. MDE first — the number that decides whether the rest is worth doing

Reviewer requirement 1, and it comes **before** the headline estimate on purpose. One
treated cluster over 14 days is very little identifying variation. We get the MDE by
permutation: plant a range of true lifts, re-simulate, and see what fraction of runs the
design detects. **If the design cannot see a lift of the size the business cares about,
a null result from it means nothing and must never be reported as "no effect".**

In [ ]:
def did_estimate(d):
    """Poisson DiD on counts. log link => the coefficient is a proportional lift.
    Adjustment set is empty by design; the fixed effects are the DESIGN, not covariates."""
    d = d.copy()
    d["date_f"] = d.booking_date.dt.strftime("%Y-%m-%d")
    m = smf.glm("member_booking_volume ~ treated + C(departure_group) + C(date_f)",
                data=d, family=sm.families.Poisson()).fit(cov_type="HC1")
    return m.params["treated"], m.bse["treated"], m.pvalues["treated"]

N_SIM = 60          # raise to 300+ if you have the time; 60 is enough to see the shape
grid = [0.00, 0.02, 0.05, 0.08, 0.12, 0.20]
power_rows = []
for lift in grid:
    hits = 0
    for i in range(N_SIM):
        r = np.random.default_rng(SEED + 1000 * i + int(lift * 1000))
        _, _, p = did_estimate(simulate(true_lift=lift, rng=r))
        hits += (p < 0.05)
    power_rows.append({"true_lift": lift, "detection_rate": hits / N_SIM})

power = pd.DataFrame(power_rows)
print(power.to_string(index=False))
detectable = power.loc[power.detection_rate >= 0.80, "true_lift"]
MDE = detectable.min() if len(detectable) else None
print()
if MDE is None:
    print("MDE > 20%: this design cannot reliably detect anything in the tested range.")
else:
    print(f"MDE (80% detection, simulated) ~ {MDE:.0%} lift in member booking volume.")
print("False-positive rate at true lift 0 should be near 0.05:",
      float(power.loc[power.true_lift == 0, 'detection_rate'].iloc[0]))

## 6. Falsifying the graph — testable implications

A graph drawn in a conversation is a hypothesis. Two implications of this one are
checkable, and the third — parallel trends — is the one everything rests on.

In [ ]:
pre = df[df.booking_date < FLIP_START].copy()

# (a) No edge flip -> departure_year: the control group must not change size in the window.
win = df[df.in_window == 1].groupby("departure_group").member_booking_volume.mean()
just_before = df[(df.booking_date < FLIP_START) &
                 (df.booking_date >= FLIP_START - pd.Timedelta(days=28))] \
                .groupby("departure_group").member_booking_volume.mean()
ctrl_move = win["depart_2027plus"] / just_before["depart_2027plus"] - 1
print(f"(a) control group volume change in the window: {ctrl_move:+.1%}")
print("    A large move here would contradict the 'no pull-forward' edge-absence and mean")
print("    the control is being depleted by the treatment.\n")

# (b) PARALLEL TRENDS on the 18-month pre-period, on the log scale.
pre["t"] = (pre.booking_date - START).dt.days / 365.0
pre["logy"] = np.log(pre.member_booking_volume.clip(lower=1))
pt = smf.ols("logy ~ t * treated_group", data=pre).fit(cov_type="HC1")
slope_gap = pt.params["t:treated_group"]
slope_p = pt.pvalues["t:treated_group"]
print(f"(b) pre-period log-trend gap between groups: {slope_gap:+.4f} per year (p={slope_p:.3f})")
print("    Near zero and insignificant is the ONLY evidence available for parallel trends.")
print("    It is evidence, never proof: the assumption is about the post-period.\n")

# (c) The arithmetic identity is NOT a causal test — state it so nobody reads it as one.
print("(c) member_booking_volume = redeeming + non-redeeming is an accounting identity.")
print("    It is checked in the assertions and carries no causal information.")

fig, ax = plt.subplots(figsize=(12, 4))
for g, sub in df.groupby("departure_group"):
    s = sub.set_index("booking_date").member_booking_volume.rolling(7).mean()
    ax.plot(s.index, np.log(s), label=g)
ax.axvspan(FLIP_START, FLIP_END, color="orange", alpha=0.3, label="flip window")
ax.set_ylabel("log(7-day mean bookings)"); ax.legend(); ax.set_title("SIMULATED — not IAGL")
plt.show()

## 7. Placebo battery — reported BEFORE the headline (requirement 6)

Fake treatment windows in the pre-period. If the design finds effects where there were
none, its finding in the real window is worth nothing.

In [ ]:
placebo_rows = []
for months_back in [3, 6, 9, 12, 15]:
    fake_start = FLIP_START - pd.DateOffset(months=months_back)
    fake_end = fake_start + pd.Timedelta(days=13)
    d = df[df.booking_date < FLIP_START].copy()
    d["treated"] = (((d.booking_date >= fake_start) & (d.booking_date <= fake_end))
                    .astype(int) * d.treated_group)
    if d.treated.sum() == 0:
        continue
    b, se, p = did_estimate(d)
    placebo_rows.append({"placebo_window": fake_start.date(),
                         "est_lift": np.expm1(b), "p": round(p, 3),
                         "flag": "FAIL" if p < 0.05 else "ok"})
placebo = pd.DataFrame(placebo_rows)
print(placebo.to_string(index=False))
n_fail = (placebo.flag == "FAIL").sum()
print(f"\nplacebo windows firing at p<0.05: {n_fail} of {len(placebo)}")
print("More than about 1 in 20 means the design manufactures effects and the headline")
print("estimate below should not be believed.")

## 8. The estimate, and the event study

Does the estimator recover the effect we planted? That, and only that, is what this cell
establishes.

In [ ]:
b, se, p = did_estimate(df)
est, lo, hi = np.expm1(b), np.expm1(b - 1.96 * se), np.expm1(b + 1.96 * se)
print(f"PLANTED lift : {TRUE_LIFT:+.2%}")
print(f"ESTIMATED    : {est:+.2%}   95% CI [{lo:+.2%}, {hi:+.2%}]   p={p:.4f}")
print(f"recovered inside the CI: {lo <= TRUE_LIFT <= hi}")

# Event study: leads test anticipation (announcement date is unknown), lags test whether
# the effect was pulled-forward demand that reverses after 14 Aug.
ev = df.copy()
ev["weeks"] = ((ev.booking_date - FLIP_START).dt.days // 7).clip(-8, 4)
ev = ev[ev.booking_date >= FLIP_START - pd.Timedelta(days=70)]
ev["wk"] = ev.weeks.astype(int)
m = smf.glm("member_booking_volume ~ C(wk, Treatment(-1)) * treated_group",
            data=ev, family=sm.families.Poisson()).fit(cov_type="HC1")
terms = [(int(k.split('[T.')[1].split(']')[0]), m.params[k], m.bse[k])
         for k in m.params.index if "wk" in k and ":treated_group" in k]
terms.sort()
fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar([t[0] for t in terms], [np.expm1(t[1]) for t in terms],
            yerr=[1.96 * t[2] for t in terms], fmt="o-")
ax.axhline(0, color="grey"); ax.axvline(-0.5, color="orange")
ax.axhline(TRUE_LIFT, color="green", ls=":", label="planted lift")
ax.set_xlabel("weeks from 1 Aug 2026"); ax.set_ylabel("lift vs control")
ax.set_title("Event study — SIMULATED"); ax.legend(); plt.show()
print("Non-zero LEADS (weeks < 0) = anticipation, and would mean the unknown announcement")
print("date matters. Negative LAGS after week 2 = pulled-forward demand, not new demand.")

## 9. The substitution check — the most likely true story

The same DiD run on redeeming and non-redeeming bookings separately. A large move in
redeeming with no move in the total is the flip changing **how holidays are paid for**,
not creating any. Under the old (wrong) reading of the data, where only redeeming
bookings were visible, that outcome would have been indistinguishable from success.

In [ ]:
def did_on(col, d=df):
    dd = d.copy(); dd["date_f"] = dd.booking_date.dt.strftime("%Y-%m-%d")
    mm = smf.glm(f"{col} ~ treated + C(departure_group) + C(date_f)",
                 data=dd, family=sm.families.Poisson()).fit(cov_type="HC1")
    return np.expm1(mm.params["treated"]), mm.pvalues["treated"]

for col in ["member_booking_volume", "avios_redeeming_bookings", "non_redeeming_bookings"]:
    e, pp = did_on(col)
    print(f"  {col:28s} {e:+7.2%}  (p={pp:.4f})")
print()
print("Read it this way: total is the finding; the split is the diagnosis. Redeeming up")
print("with total flat = substitution. Both up together = genuine incremental volume.")

## 10. Result summary — paste this back

Copy the whole block below into the chat.

In [ ]:
print("=" * 72)
print("q-0001 DAG ASSESSMENT — data_mode: SIMULATED")
print("NO NUMBER HERE IS ABOUT IAGL. It tests the graph and the estimator only.")
print("=" * 72)
print(f"seed                     : {SEED}")
print(f"rows / panel grain       : {len(df)} (booking_date x departure_group, daily)")
print(f"pre-period               : {(FLIP_START - df.booking_date.min()).days} days")
print()
print("ASSERTIONS")
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print()
print("GRAPH")
print(f"  acyclic                : yes ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")
print(f"  adjustment set         : {{}} (empty, by design)")
print(f"  unobserved nodes       : {[n for n, o in OBSERVED.items() if not o]}")
print(f"  pre-trend gap (log/yr) : {slope_gap:+.4f}  p={slope_p:.3f}")
print(f"  control-group move     : {ctrl_move:+.1%}")
print()
print("POWER")
print(f"  MDE @80% (simulated)   : {'>20%' if MDE is None else f'{MDE:.0%}'}")
print(f"  false positives @ lift 0: {float(power.loc[power.true_lift == 0, 'detection_rate'].iloc[0]):.2f}")
print()
print("PLACEBOS")
print(f"  windows firing p<0.05  : {n_fail} of {len(placebo)}")
print()
print("RECOVERY")
print(f"  planted lift           : {TRUE_LIFT:+.2%}")
print(f"  estimated lift         : {est:+.2%}  95% CI [{lo:+.2%}, {hi:+.2%}]  p={p:.4f}")
print(f"  planted inside CI      : {lo <= TRUE_LIFT <= hi}")
print("=" * 72)